# F1 Stage A — Data Access & Reference Numbers Check (`01_data_access_check.ipynb`)

Verifies all data sources from `docs/plans/phases/F1_data_collection.md` (Section 10.1 checklist).

In [1]:
import json

import numpy as np
import pandas as pd

from marketsim import paths
from marketsim.data import api

raw = paths.raw_dir()
cal = api.load_calendar()
print(f'Master calendar: {len(cal)} trading days ({cal.index.min().date()} .. {cal.index.max().date()})')

Master calendar: 1255 trading days (2021-01-04 .. 2025-12-31)


## 1. Summary Table: Source -> Downloaded -> Reference Numbers Matched -> Comment

In [2]:
rows = []

# 1. FRED daily
for sid, exp_miss in [('DCOILWTICO', 56), ('DCOILBRENTEU', 41), ('DHHNGSP', 55), ('SP500', 49)]:
    df = pd.read_csv(raw / 'fred' / f'{sid}.csv')
    df['dt'] = pd.to_datetime(df.iloc[:, 0])
    df['val'] = pd.to_numeric(df.iloc[:, 1], errors='coerce')
    sub = df[(df['dt'] >= '2021-01-01') & (df['dt'] <= '2025-12-31')]
    n_rows = len(sub)
    n_miss = int(sub['val'].isna().sum())
    rows.append({
        'source': f'FRED {sid}',
        'downloaded': 'YES',
        'numbers_matched': n_rows == 1304 and n_miss == exp_miss,
        'comment': f'2021-2025: {n_rows} rows, {n_miss} missing (expected 1304, {exp_miss})'
    })

# 2. FRED monthly copper
cop_m = api.load_commodities_monthly()
rows.append({
    'source': 'FRED PCOPPUSDM (monthly copper)',
    'downloaded': 'YES',
    'numbers_matched': len(cop_m) == 60,
    'comment': f'{len(cop_m)} monthly rows in 2021-2025'
})

# 3. Yahoo stocks
adj = api.load_stock_prices(adjusted=True)
raw_p = api.load_stock_prices(adjusted=False)
duk_diff = abs(float(raw_p.loc['2021-01-04', 'DUK']) - float(adj.loc['2021-01-04', 'DUK'])) > 10
rows.append({
    'source': 'Yahoo 18 Stocks (adj & raw)',
    'downloaded': 'YES',
    'numbers_matched': adj.shape == (1255, 18) and int(adj.isna().sum().sum()) == 0 and duk_diff,
    'comment': f'{adj.shape[0]}x{adj.shape[1]}, 0 NaNs; DUK 2021-01-04 raw={raw_p.loc["2021-01-04", "DUK"]:.2f} vs adj={adj.loc["2021-01-04", "DUK"]:.2f}'
})

# 4. Yahoo futures vs World Bank
comm_d = api.load_commodities_daily()
joined_m = pd.concat([comm_d.resample('MS').mean(), cop_m], axis=1).dropna()
c_gold = joined_m['gold'].corr(joined_m['wb_gold'])
c_plat = joined_m['platinum'].corr(joined_m['wb_platinum'])
c_cop = joined_m['copper_usd_per_lb'].corr(joined_m['wb_copper_usd_per_mt'])
rows.append({
    'source': 'Yahoo 5 Futures vs World Bank',
    'downloaded': 'YES',
    'numbers_matched': c_gold > 0.999 and c_plat > 0.995 and c_cop > 0.93,
    'comment': f'Correlations with WB: Gold={c_gold:.4f}, Platinum={c_plat:.4f}, Copper={c_cop:.3f}'
})

# 5. French 49 & Market factor
ff49 = api.load_ff49()
mkt = api.load_market_daily()
sp_ff_corr = np.log(mkt['sp500']).diff().corr(mkt['ff_mkt_ret'])
rows.append({
    'source': 'French 49 Industries & Mkt Factor',
    'downloaded': 'YES',
    'numbers_matched': ff49.shape == (1255, 49) and sp_ff_corr > 0.99,
    'comment': f'FF49 shape={ff49.shape}, 0 NaNs; corr(SP500, Mkt-RF+RF)={sp_ff_corr:.4f}'
})

# 6. BEA 2023
a23 = api.load_io_matrix(2023)
ind23 = api.load_io_industries(2023)
rho = float(np.max(np.abs(np.linalg.eigvals(a23.to_numpy()))))
rows.append({
    'source': 'BEA Use Table (2022 & 2023)',
    'downloaded': 'YES',
    'numbers_matched': a23.shape == (71, 71) and abs(a23.loc['324', '481'] - 0.1292) < 1e-3 and abs(a23.loc['211', '324'] - 0.5077) < 1e-3,
    'comment': f'71x71; A[324,481]={a23.loc["324","481"]:.4f}, A[211,324]={a23.loc["211","324"]:.4f}, out[481]={ind23.loc["481","output"]:.0f}, rho={rho:.3f}'
})

# 7. EDGAR SIC vs companies.csv
comp = api.load_companies()
edgar_ok = True
for r in comp.itertuples():
    p = raw / 'edgar' / 'submissions' / f'CIK{int(r.cik):010d}.json'
    s = json.loads(p.read_text(encoding='utf-8'))
    if str(s['sic']) != str(r.sic):
        edgar_ok = False
rows.append({
    'source': 'SEC EDGAR SIC (18 companies)',
    'downloaded': 'YES',
    'numbers_matched': edgar_ok and len(comp) == 18,
    'comment': 'All 18 CIK submissions downloaded; SIC codes match data/mapping/companies.csv 100%'
})

summary_df = pd.DataFrame(rows)
summary_df

,source,downloaded,numbers_matched,comment
0,FRED DCOILWTICO,YES,True,"2021-2025: 1304 rows, 56 missing (expected 130..."
1,FRED DCOILBRENTEU,YES,True,"2021-2025: 1304 rows, 41 missing (expected 130..."
2,FRED DHHNGSP,YES,True,"2021-2025: 1304 rows, 55 missing (expected 130..."
3,FRED SP500,YES,True,"2021-2025: 1304 rows, 49 missing (expected 130..."
4,FRED PCOPPUSDM (monthly copper),YES,True,60 monthly rows in 2021-2025
5,Yahoo 18 Stocks (adj & raw),YES,True,"1255x18, 0 NaNs; DUK 2021-01-04 raw=89.71 vs a..."
6,Yahoo 5 Futures vs World Bank,YES,True,"Correlations with WB: Gold=0.9999, Platinum=0...."
7,French 49 Industries & Mkt Factor,YES,True,"FF49 shape=(1255, 49), 0 NaNs; corr(SP500, Mkt..."
8,BEA Use Table (2022 & 2023),YES,True,"71x71; A[324,481]=0.1292, A[211,324]=0.5077, o..."
9,SEC EDGAR SIC (18 companies),YES,True,All 18 CIK submissions downloaded; SIC codes m...


## 2. F2 Stage A Check: Chain Weights (WTI -> COP -> VLO -> DAL)

In [3]:
ret = api.load_returns_daily().loc['2021-01-01':'2023-12-31'].dropna()

def ols(y, X):
    Xm = np.column_stack([np.ones(len(X)), X.to_numpy()])
    b, *_ = np.linalg.lstsq(Xm, y.to_numpy(), rcond=None)
    resid = y.to_numpy() - Xm @ b
    s2 = resid @ resid / (len(y) - Xm.shape[1])
    se = np.sqrt(np.diag(s2 * np.linalg.inv(Xm.T @ Xm)))
    r2 = 1 - resid.var() / y.to_numpy().var()
    return dict(zip(X.columns, np.round(b[1:], 3))), dict(zip(X.columns, np.round(b[1:] / se[1:], 1))), round(float(r2), 2)

for label, child, parents in [
    ('COP direct=full', 'COP', ['wti', 'sp500']),
    ('EOG direct', 'EOG', ['wti', 'sp500']),
    ('VLO direct', 'VLO', ['wti', 'sp500']),
    ('VLO full', 'VLO', ['COP', 'wti', 'sp500']),
    ('DAL direct', 'DAL', ['wti', 'sp500']),
    ('DAL full', 'DAL', ['VLO', 'wti', 'sp500']),
]:
    w, t, r2 = ols(ret[child], ret[parents])
    print(f'{label:16s} | weights={w} | t={t} | R2={r2}')

COP direct=full  | weights={'wti': np.float64(0.516), 'sp500': np.float64(0.619)} | t={'wti': np.float64(20.4), 'sp500': np.float64(10.9)} | R2=0.44
EOG direct       | weights={'wti': np.float64(0.553), 'sp500': np.float64(0.622)} | t={'wti': np.float64(18.9), 'sp500': np.float64(9.5)} | R2=0.4
VLO direct       | weights={'wti': np.float64(0.43), 'sp500': np.float64(0.603)} | t={'wti': np.float64(14.1), 'sp500': np.float64(8.9)} | R2=0.29
VLO full         | weights={'COP': np.float64(0.701), 'wti': np.float64(0.069), 'sp500': np.float64(0.169)} | t={'COP': np.float64(19.6), 'wti': np.float64(2.2), 'sp500': np.float64(2.8)} | R2=0.53
DAL direct       | weights={'wti': np.float64(-0.043), 'sp500': np.float64(1.274)} | t={'wti': np.float64(-1.5), 'sp500': np.float64(19.6)} | R2=0.34
DAL full         | weights={'VLO': np.float64(0.222), 'wti': np.float64(-0.139), 'sp500': np.float64(1.14)} | t={'VLO': np.float64(6.5), 'wti': np.float64(-4.4), 'sp500': np.float64(17.1)} | R2=0.37
